**To the student**: summarize in your own words, based on the Wikipeida content quoted above, and potentially additional sources:
* What are we trying to solve in a classification problem?
* What is a classifier?

**Answer**: 
* In a classification problem, we are trying to assign labels or categories to data points/observations based on their features. The goal is to accurately predict the category of new, unseen data based on patterns learned from a training dataset.
* A classifier is a model or algorithm that is trained to recognize patterns in the data and make predictions about which category a given data point belongs to. It takes input features and outputs a predicted label, effectively serving as a decision-making tool that helps in categorizing data into predefined classes.

**To the student**: 
* What does the Boolean values (`True` and `False`) stand for in the target `y_train_5`?
* Why wasn't `X` changed as well?

**Answer**:
* The Boolean values `True` and `False` in the target `y_train_5` represent whether a given data point belongs to a specific class (e.g., "5") or not. `True` indicates that the data point is of the class being predicted, while `False` indicates that it is not.
* `X` wasn't changed because it contains the input features of the data points, which remain the same regardless of the target class being predicted. The features are used to train the classifier, while the target variable (`y_train_5`) is modified to reflect the specific classification task at hand.

**To the student**: 
* Which image does `some_digit` stand for? (scroll above if needed)
* What is the meaning of the resulting Boolean value (`True` and `False`)?

**Answer**:
* Some_digit represents the image of the digit "4" from the dataset. It is a specific data point that we are using to test our classifier.
* The output False means the classifier predicted that some_digit belongs to the negative class — i.e., the classifier predicted that the image is not a 5.

**To the student**: Regarding the `cross validate` function, browse the sklearn's [docs](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#sklearn.model_selection.cross_validate) for `cross_validate``:
* Describe in your own words what the `cv` argument is for.
* How many times has the model been trained?
* How is the value of `cv` related to the output of this function?
* What do `fit_time`,  `score_time` and `test_score` stand for?

**Answers**
* It specifies the cross-validation splitting strategy. Setting cv=3 partitions the training dataset into 3 equal stratified subsets (folds). In 3 consecutive rounds, 2 folds are combined to train the model, and the remaining fold is used as the evaluation set.

* The model was trained 3 times (once per fold).

* The value of cv determines the length of each array in the returned dictionary. Since cv=3, each array (fit_time, score_time, test_score) contains exactly 3 values, corresponding to the results from each of the 3 iterations.

* What do the three arrays, fit_time, score_time and test_score stand for?
    - `fit_time`: The time (in seconds) taken to train the estimator on the training folds for each CV split.
    - `score_time`: The time (in seconds) taken to compute predictions and evaluate the scoring metric on the validation fold for each split.
    - `test_score`: The resulting performance score (here, classification accuracy) evaluated on each validation fold.


**To the student**: Regrading the accuracy
* Could you provide a textual description of the accuracy result for the first fold? For instance "out of 100 digits, XYZ were correctly predicted, while ZYX were incorrectly predicted"
* Given the uniform distirbution of samples across all 10 digits, and given that the classifier classify one of them versus all the others (`5` versus all others in our case), what would be a simple handwritten rule to achieve 90% accuracy?
* How would you describe the goodness of classification of the `SGDClassifier` versus the expected accuracy of such a handwritten rule? Please describe it in a sentence or two. 


**Answer**:
* The first fold's accuracy is `0.95035`. With `cv=3`, each validation fold holds 60,000 / 3 = 20,000 digits. So **out of 20,000 digits, 19,007 were correctly predicted (5 or not 5), while 993 were incorrectly predicted**. Per 100 digits, that is about 95 correct and 5 wrong.
* Because the 10 digits are roughly evenly distributed, only about 10% of the images are 5s and about 90% are not. A simple handwritten rule is: **always predict "not 5" (`False`)**, whatever the image. It is right for every non-5 image, so it reaches about 90% accuracy. Here it is actually 54,579 / 60,000 ≈ 91%, since 5s make up a little under 10% of the training set.
* The `SGDClassifier` (~95–96% accuracy) beats the "always not 5" rule (~90–91%), but only by about 5 percentage points. Put another way, it cuts the error rate roughly in half (from ~9% to ~5%). Because a rule that never looks at the image already scores 90%, accuracy is a misleading measure on this imbalanced task, and the 95% looks much better than it is.

**To the student**: 
* What is the accuracy of the `DummyClassifier`? 
* How does it compare to the handwritten rule you suggested above, and to the `SGDClassifier`? 

**Answer**:
* The `DummyClassifier` scores an accuracy of **`0.90965` (~91%) in all three folds**. It ignores `X` and always predicts the most frequent class, `False` ("not 5"), as shown by `any(dummy_clf.predict(X_train))` returning `False`.
* It is exactly the handwritten "always predict not 5" rule suggested above, implemented in `sklearn`, so it gets the same ~91% accuracy, the share of non-5s in the data (54,579 / 60,000). The `SGDClassifier` (~95–96%) does only about 4–5 points better. The dummy model shows that high accuracy alone means little on skewed data: a model that never finds a single 5 still gets 91%. We need metrics that focus on the positive class, such as the confusion matrix, precision and recall.

**To the student**: 
## Exercise - Fashion-MNIST data set

Follow the same process we used to analyze the MNIST data set, but this time do it using the **Fashion-MNIST** data set.

- Get the data set (code cell below)
- Preprocess the data set
- Train a model
- Evaluate the model

> Are you able to replicate the process? Provide your insights.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.linear_model import SGDClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# 1. Get the data set
fashion_mnist = fetch_openml('Fashion-MNIST', as_frame=False, parser='auto')
X_fashion = fashion_mnist.data
y_fashion = fashion_mnist.target

# 2. Preprocess: same train/test split as MNIST, and a binary target
#    label '9' = Ankle boot (positive class) vs. all other clothing items
X_train_f, X_test_f = X_fashion[:60000], X_fashion[60000:]
y_train_f, y_test_f = y_fashion[:60000], y_fashion[60000:]
y_train_f_9 = (y_train_f == '9')
y_test_f_9 = (y_test_f == '9')

# 3. Train a model
sgd_clf_f = SGDClassifier(random_state=42)
sgd_clf_f.fit(X_train_f, y_train_f_9)

# 4. Evaluate the model
print("SGD accuracy:  ", cross_validate(sgd_clf_f, X_train_f, y_train_f_9, cv=3, scoring="accuracy")["test_score"])
print("Dummy accuracy:", cross_validate(DummyClassifier(), X_train_f, y_train_f_9, cv=3, scoring="accuracy")["test_score"])

y_train_f_pred = cross_val_predict(sgd_clf_f, X_train_f, y_train_f_9, cv=3)
print(confusion_matrix(y_train_f_9, y_train_f_pred))
print("Precision:", precision_score(y_train_f_9, y_train_f_pred))
print("Recall:   ", recall_score(y_train_f_9, y_train_f_pred))
print("F1:       ", f1_score(y_train_f_9, y_train_f_pred))

**Answer**:

Yes, the process carries over directly. Fashion-MNIST has the same format as MNIST: 70,000 grayscale 28×28 images (784 pixel features), 10 classes with 7,000 images each, and string labels `'0'`–`'9'`. The same code works almost unchanged: fetch with `fetch_openml`, split into the first 60,000 for training and the last 10,000 for testing, build a Boolean one-vs-all target, train an `SGDClassifier`, and evaluate it with `cross_validate` / `cross_val_predict`.

Results (3-fold CV on the training set, `random_state=42`):

| Positive class | SGD accuracy | Dummy accuracy | Confusion matrix `[[TN, FP], [FN, TP]]` | Precision | Recall | F1 |
|---|---|---|---|---|---|---|
| `'9'` Ankle boot | 0.977 / 0.979 / 0.978 | 0.90 | `[[53202, 798], [512, 5488]]` | 0.873 | 0.915 | 0.893 |
| `'0'` T-shirt/top | 0.948 / 0.950 / 0.952 | 0.90 | `[[52869, 1131], [1871, 4129]]` | 0.785 | 0.688 | 0.733 |

Insights:
* **The accuracy trap is the same.** Each class is exactly 10% of the data, so the `DummyClassifier` ("always negative") scores exactly 90%. SGD's 95–98% accuracy has to be judged against that 90% baseline, not against 0%.
* **Difficulty depends on which class is positive.** "Ankle boot" has a distinctive shape that differs from most other items, so it is easy to separate (F1 ≈ 0.89). "T-shirt/top" is often confused with "Shirt", "Pullover" and "Dress", which have similar silhouettes, so recall falls to ~0.69 and F1 to ~0.73, even though accuracy still looks high (~95%).
* Overall accuracy hides these differences, while precision, recall and F1 show them. This makes the case for the confusion-matrix-based metrics that follow.

To the student: Using `sklearn` resources about Confusion Matrix: [docs](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) and [user guide](https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix):

* What are the required arguments that the `confusion_matrix` function expects?
* What is the output of this function, specifically for a binary classifier such as the one tested here?
* What is the meaning of each value in both of the confusion matrics returned above?

**Answer**:
* **Required arguments:** `y_true` (the ground-truth target labels) and `y_pred` (the labels predicted by the classifier), two array-likes of the same length. Optional arguments include `labels` (which classes to include and in what order), `sample_weight` and `normalize` (`'true'`, `'pred'` or `'all'` to show fractions instead of counts).
* **Output:** a NumPy array of shape `(n_classes, n_classes)`. Entry `[i, j]` is the number of samples whose **true** class is `i` and whose **predicted** class is `j`, so rows are actual classes and columns are predicted classes, in sorted label order. For a binary classifier with labels `False`/`True` this is a 2×2 matrix:

  |  | predicted `False` (not 5) | predicted `True` (5) |
  |---|---|---|
  | **actual `False` (not 5)** | TN | FP |
  | **actual `True` (5)** | FN | TP |

* **Meaning of the values:**
  * `SGDClassifier` predictions, `[[53892, 687], [1891, 3530]]`:
    * **TN = 53,892**: non-5 images correctly classified as "not 5".
    * **FP = 687**: non-5 images wrongly classified as "5" (false alarms).
    * **FN = 1,891**: actual 5s the model missed (classified as "not 5").
    * **TP = 3,530**: actual 5s correctly classified as "5".
    * All four add up to 60,000. The model catches 3,530 of the 5,421 real 5s.
  * Perfect predictions, `[[54579, 0], [0, 5421]]`:
    * **TN = 54,579**: all non-5 images are correctly rejected.
    * **TP = 5,421**: all 5s are correctly detected.
    * **FP = FN = 0**: no errors at all.
    * A perfect classifier has non-zero values only on the main diagonal. The diagonal also shows the true class sizes: 54,579 non-5s and 5,421 fives.

**To the student**: 
#### Exercise - Calculate Precision and Recall while evaluating an Autonomous Security Drone

A company has developed an AI-powered security drone to patrol a large warehouse at night. Its primary task is to use its camera to identify if a person is present in a restricted area (the relevant element).

During a simulated test, the drone flies through the warehouse 500 times, analyzing a different camera view each time. The test includes staged scenarios to check the AI's accuracy.

Here is a summary of the test results:

Throughout the 500 views, there were a total of 10 instances where a person was actually present.
The drone's AI system sounded the alarm, claiming it saw a person, 12 times in total.
Upon reviewing the drone's decisions, it was found that of the 12 alarms, 8 of them were correct (a person was truly there). This means 4 alarms were false alarms, triggered by things like shadows or falling boxes.

These are your tasks:

> Calculate the Metrics: Based on the information in the scenario, calculate the Precision and the Recall of the security drone's detection system. Please show the fractions you used for your calculations.

> Explain Your Results (talking points): In your own words, explain what your calculated values for Precision and Recall mean.

**Answer**:

From the scenario (500 camera views):
* **TP** (alarm, person really there) = 8
* **FP** (alarm, no person: false alarm) = 12 − 8 = 4
* **FN** (person there, no alarm: missed) = 10 − 8 = 2
* **TN** (no person, no alarm) = 500 − 8 − 4 − 2 = 486

**Precision** = TP / (TP + FP) = 8 / (8 + 4) = **8 / 12 ≈ 0.667 (66.7%)**

**Recall** = TP / (TP + FN) = 8 / (8 + 2) = **8 / 10 = 0.80 (80%)**

Talking points:
* **Precision (66.7%):** when the drone sounds the alarm, a person is really there about 2 times out of 3. The other third (4 of 12 alarms) are false alarms caused by shadows, falling boxes and similar. Security staff would be sent out for nothing about once every three alarms, which wastes time and may lead them to start ignoring alarms.
* **Recall (80%):** of all the times an intruder was actually present, the drone detected 8 out of 10. It missed 2 out of 10 real intruders, a serious gap for a security system.
* **Trade-off:** for a security drone, a missed intruder (FN) is usually much more costly than a false alarm (FP). The company would likely prioritize **raising recall**, for example by lowering the detection threshold, even if precision drops a bit further.
* **Why not accuracy:** accuracy would be (8 + 486) / 500 = 98.8%, which sounds excellent but hides both the missed intruders and the false alarms. That is the same accuracy trap seen with MNIST.

**To the student**: 
* Similar to the manual calculation of precision, in a new cell below, write down the Python code for calculating recall

**To the student**: 
* Using the [definition](https://en.wikipedia.org/wiki/F-score#Definition) calculate the F1 score that was generated above, using True Positives (TP), False Negatives (FN) and False Positives (FP). Write this as a Python code in a new cell below.

**To the student**: 
* Assume that you are developing an ML classifeir for medical diagnostics, with the propose of being a first indicator to identify whether a patient has a medical condition that requires treatment or not. Based on this result, the data wil be delivered to a medical professional or not (this might not be a great practice, but let's assume it for the sake of the example). What would be better, a high precision or a high recall? 
* Likewise, what would be better to optimize for (a high precision versus recall), when an ML classifier should highlight the best investment opportunities in a stock exchange?
* Add two more real-world cases in which:
  * Would a high recall will be desired, even on the expense of a lower precision? 
  * And when would a high precision will be preferred, on the expense of a lower recall? 


**To the student**: 
* How do precision and recall change with increasing or decreasing the threshold? 
* What is the intuition behind this behavior?